# 9) CNNs on MNIST dataset: Data Augmentation

>**Data Augmentation**: increasing the amount of avaiable data, by adding to the original dataset the same data after applying a transformation such as
>- translation
>- rotation
>- zoom

*Advantages*:
- fix the problem of small datasets;
- make the CNN (enough) invariant for these transformations.

In general, also the accuracy increases.

In our context, MNIST is a small dataset and we apply data augmentation.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import MNIST

from tqdm import tqdm

In [4]:
from torchvision.transforms import v2                   # TRANSFORM IMAGES: TRANSLATIONS, ROTATIONS, ZOOM

from torchvision.transforms import InterpolationMode    # interpolation method to rotate

---

## Prepare MNIST dataset as usual

Dataset, Subsets, Dataloader.

In [8]:
class MNISTDataset(Dataset):
    def __init__(self):
        data = MNIST('mnist', download=True)
        self.images, self.labels = data.data, data.targets

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx].float() / 255.0
        label = self.labels[idx]
        return image, label

dataset = MNISTDataset()

idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)
train_dataset = Subset(dataset, idx_train)
valid_dataset = Subset(dataset, idx_valid)

In [9]:
def collate_fn(batch):
    images = []
    labels = []
    for image, label in batch:
        images.append(image)
        labels.append(label)
    images = torch.stack(images, dim=0)
    labels = torch.tensor(labels)
    return images, labels

train_dataloader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 0,
    drop_last = True,
    collate_fn = collate_fn
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 0,
    drop_last = True,
    collate_fn = collate_fn
)

## Define an easy CNN model

In [6]:
model = nn.Sequential(
    nn.Conv2d(1, 32, 3),      # 1st convolutional layer
    nn.MaxPool2d(2),          # max pooling
    nn.ReLU(),                # activation function

    nn.Conv2d(32, 64, 3),     # 2nd convolutional layer
    nn.Flatten(),             # flatten
    nn.ReLU(),                # activation function

    nn.Linear(7744, 10)       # output
)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [7]:
import torchinfo
torchinfo.summary(model)

Layer (type:depth-idx)                   Param #
Sequential                               --
├─Conv2d: 1-1                            320
├─MaxPool2d: 1-2                         --
├─ReLU: 1-3                              --
├─Conv2d: 1-4                            18,496
├─Flatten: 1-5                           --
├─ReLU: 1-6                              --
├─Linear: 1-7                            77,450
Total params: 96,266
Trainable params: 96,266
Non-trainable params: 0

## Training and Evaluation

In [8]:
for epoch in range(10):

    for images, labels in tqdm(train_dataloader, leave=False):
        optimizer.zero_grad()
        y_pred = model(images.view(images.shape[0], 1, 28, 28))   # NEED TO SPECIFY THE SIZE OF IMAGES
        loss_result = loss_function(y_pred, labels)
        loss_result.backward()
        optimizer.step()

    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in valid_dataloader:
            logits = model(images.view(images.shape[0], 1, 28, 28))
            all_preds.extend(torch.argmax(logits, dim=1))
            all_labels.extend(labels)
    print(accuracy_score(all_labels, all_preds))

0.9852941176470589


0.9817847593582888


0.9896390374331551


0.9894719251336899


0.9908088235294118


0.9896390374331551


0.9893048128342246


0.986798128342246


0.9898061497326203


0.9893048128342246


A simple model as this is able to obtain almost the 99% of accuracy.

---

##  Data Augmentation of MNIST

In [5]:
class MNISTDataset(Dataset):

    def __init__(self, do_augmentation=False):

        data = MNIST('mnist', download=True)
        self.images, self.labels = data.data, data.targets

        # AUGMENTATION BLOCK
        self.do_augmentation = do_augmentation
        self.translation = v2.RandomCrop(size=28, padding=4, fill=0)
        self.rotation = v2.RandomRotation(degrees=10, fill=0, interpolation=InterpolationMode.BILINEAR)
        self.zoom = v2.RandomResizedCrop(size=(28, 28), scale=[0.8, 1.2], antialias=True)

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx: int):        # return element of index i

        image = self.images[idx].float() / 255.0
        label = self.labels[idx]

        # AUGMENTATION BLOCK
        if self.do_augmentation:
            image = self.translation(image.view(1, 28, 28))
            image = self.rotation(image)
            image = self.zoom(image)
        
        return image[0], label


In [6]:
dataset = MNISTDataset(do_augmentation=True)

idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)
train_dataset = Subset(dataset, idx_train)
valid_dataset = Subset(dataset, idx_valid)

In [7]:
def collate_fn(batch):
    # batch : une liste de longueur 32 = batch_size, avec batch[0] = premiere_image, premiere_etiquette, etc.
    images = []
    labels = []
    for image, label in batch:
        images.append(image)
        labels.append(label)
    # Deux listes : images et labels, chacune de longueur 32 de éléments : torch.Tensor
    # stack "empile" les 32 tenseurs dans la dimension 0
    images = torch.stack(images, dim=0)  # taille : (32, 784)
    labels = torch.tensor(labels)  # taille : (32,)
    return images, labels

train_dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    drop_last=True,
    collate_fn=collate_fn
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=32,
    # Avec shuffle=False, on a les éléments de validation
    # toujours dans le même ordre ; cela ne change pas l'accuracy,
    # mais c'est utile par exemple si on veut toujours visualiser
    # les premiers points de validation, etc.
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    drop_last=False
)

In [8]:
model = nn.Sequential(
    nn.Conv2d(1, 32, 3),      # 1st convolutional layer
    nn.MaxPool2d(2),          # max pooling
    nn.ReLU(),                # activation function

    nn.Conv2d(32, 64, 3),     # 2nd convolutional layer
    nn.Flatten(),             # flatten
    nn.ReLU(),                # activation function

    nn.Linear(7744, 10)       # output
)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(10):
    model.train()

    for images, labels in train_dataloader:
        if images.ndim == 3:
            images = images.unsqueeze(1)  # [B, 28, 28] -> [B, 1, 28, 28]
            
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        y_pred = model(images)
        loss_result = loss_function(y_pred, labels)
        loss_result.backward()
        optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in valid_dataloader:
            if images.ndim == 3:
                images = images.unsqueeze(1)
                
            images = images.to(device)
            
            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch + 1}: accuracy = {accuracy:.4f}")

Epoch 1: accuracy = 0.9115
Epoch 2: accuracy = 0.9423
Epoch 3: accuracy = 0.9455
Epoch 4: accuracy = 0.9342
Epoch 5: accuracy = 0.9567
Epoch 6: accuracy = 0.9593
Epoch 7: accuracy = 0.9615
Epoch 8: accuracy = 0.9645
Epoch 9: accuracy = 0.9628
Epoch 10: accuracy = 0.9713
